# Exercise 1: Tensor basics 
In this exercise you will learn the basics of tensor creation, manipulation, indexing, broadcasting, vectorization, einsum, and attention masking fundamentals. These basics are important for understanding any complex implementation later on so make sure you understand them well.

**To complete this exercise fill in all TODOs in the functions below.** 

Make sure to check the output of your function and whether or not it fulfills the requirements outlined in the function definition. Do NOT change the function signature or name since we will be running checks on your functions during grading.

### Shape legend used in this notebook
- `B`: batch size
- `T`: sequence length / time
- `D`: feature dimension
- `H`: number of attention heads
- `Dh`: per-head feature dimension

### Debugging tip: what to print
When you get a shape error, print:
- `x.shape`, `x.dtype`, `x.device`
- `x.is_contiguous()` (important for `view`)
For masks also print:
- `mask.shape`, `mask.dtype`, `mask.sum()` and a small slice like `mask[0, :10]`

### Reproducibility tip: seeding in PyTorch
Many operations in deep learning involve randomness (e.g., initializing model weights, shuffling data, dropout, random augmentations).
**Seeding** sets the starting state of PyTorch’s random number generator so that these random choices become **repeatable**.

- If you set the same seed and run the same code again, you should get the same *random* tensors / initial weights.
- If you don’t set a seed, results can vary between runs.

Common usage: `torch.manual_seed(seed)`

Note: even with fixed seeds, some GPU operations can still be non-deterministic due to performance optimizations. For this assignment, seeding is mainly to make debugging easier and to ensure everyone can reproduce the same intermediate results. If you are given a seed, make sure to use it when creating tensors or performing other operations.

## Tensor creation
This warmup exercise teaches you how to create tensors with different shapes and values. A few details about tensor creation that are good to know:
- `torch.tensor([...])` infers dtype from Python values (ints → integer tensor, floats → float tensor).
- `torch.arange(start, end)` is **end-exclusive**.
- `torch.linspace(start, end, steps)` is **end-inclusive**.

In [2]:
import torch #missing imports due to error

def make_tensor(data, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """ Create a tensor from Python data (list/tuple/nested lists). """
    
    return torch.tensor(data, dtype=dtype, device = device)
    

x = make_tensor([[1, 2], [3, 4]], dtype=torch.float32)

print(x)

tensor([[1., 2.],
        [3., 4.]])


In [3]:
from typing import Sequence #missing import due to error

def make_zeros(shape: Sequence[int], dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with zeros."""
    
    return torch.zeros(shape, dtype=dtype, device=device)

z = make_zeros((2, 3), dtype=torch.float64)

print(z)

tensor([[0., 0., 0.],
        [0., 0., 0.]], dtype=torch.float64)


In [4]:
def make_ones_like(x: torch.Tensor) -> torch.Tensor:
    """Create a tensor of ones with the same shape, dtype, and device as x. """

    return torch.ones_like(x)

base = torch.randn(2, 3, dtype=torch.float32)
ones = make_ones_like(base)

print(base, ones)

tensor([[ 0.0954, -1.5856,  1.4757],
        [ 0.2372, -0.5160,  0.6860]]) tensor([[1., 1., 1.],
        [1., 1., 1.]])


In [5]:
def make_arange(start: int, end: int, step: int = 1, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor containing values [start, start+step, ..., < end]."""
    
    return torch.arange(start=start, end=end, step=step, dtype=dtype, device=device)

ar = make_arange(0, 5, 2, dtype=torch.int64)

print(ar)

tensor([0, 2, 4])


In [6]:
def make_linspace(start: float, end: float, steps: int, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor with evenly spaced values from start to end (inclusive)."""
    
    return torch.linspace(start=start, end=end, steps=steps, dtype=dtype, device=device)

ls = make_linspace(0.0, 1.0, steps=5, dtype=torch.float32)

print(ls)

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


In [7]:
def make_randn(shape: Sequence[int], seed: int | None = None, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with values from a standard normal distribution."""
    
    
    torch.manual_seed(seed)
    return torch.randn(shape, dtype=dtype, device=device)

a = make_randn((2, 3), seed=123, dtype=torch.float32)

print(a)

tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093]])


In [8]:
def cast_dtype_and_move(x: torch.Tensor, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Convert tensor dtype and move to device."""
    
    return x.to(device=device, dtype=dtype)

casted = cast_dtype_and_move(torch.tensor([1, 2, 3]), torch.device("cpu"), torch.float32)

print(casted)

tensor([1., 2., 3.])


## Shape manipulation
Now that we covered the basic tensor creation schemes, we want to focus on shape manipulation. Understanding the difference between these mechanisms is key for building larger systems and many people still get it wrong. 
The core ideas to understand are:
- **Contiguous tensors** store data in a single, row-major memory layout.
- Many ops (especially slicing like `x[:, ::2]`, `transpose`, `permute`) often create **non-contiguous** tensors (no copy but different strides).
- `view(...)` is **zero-copy** but typically requires **contiguous** memory → may throw an error.
- `reshape(...)` tries to return a view, but if the tensor is non-contiguous it will **allocate/copy**.
- `contiguous()` forces a contiguous copy when the tensor isn’t contiguous.

If you *need* a view after reordering dims: call `x = x.contiguous()` first (this makes a contiguous copy).

In [9]:
def reshape_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """Reshape tensor to new_shape (may return a view or a copy)."""
    
    return x.reshape(new_shape)

x = torch.arange(6)
y = reshape_tensor(x, (2, 3))

print(x, y)

tensor([0, 1, 2, 3, 4, 5]) tensor([[0, 1, 2],
        [3, 4, 5]])


In [10]:
def view_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """View tensor as new_shape (requires contiguous memory and doesn't allocate new memory for the tensor data)."""
    
    return x.view(new_shape)

y_view = view_tensor(x, (2, 3))

print(x, y_view)

tensor([0, 1, 2, 3, 4, 5]) tensor([[0, 1, 2],
        [3, 4, 5]])


In [11]:
def flatten_from_dim(x: torch.Tensor, start_dim: int = 0) -> torch.Tensor:
    """Flatten a tensor starting from start_dim into a single dimension."""
    
    return torch.flatten(x, start_dim=start_dim)

x2 = torch.randn(2, 3, 4)
flat = flatten_from_dim(x2, start_dim=1)

print(x2)
print(flat)

tensor([[[ 0.2403, -0.5516, -0.5697,  1.0076],
         [-0.0770, -1.0205, -0.1690,  0.9178],
         [-0.3885, -0.9343, -0.4991, -1.0867]],

        [[ 0.9624,  0.2492, -0.4845, -2.0929],
         [ 0.0983, -0.0935,  0.2662, -0.5850],
         [-0.3430, -0.6821, -0.9887, -1.7018]]])
tensor([[ 0.2403, -0.5516, -0.5697,  1.0076, -0.0770, -1.0205, -0.1690,  0.9178,
         -0.3885, -0.9343, -0.4991, -1.0867],
        [ 0.9624,  0.2492, -0.4845, -2.0929,  0.0983, -0.0935,  0.2662, -0.5850,
         -0.3430, -0.6821, -0.9887, -1.7018]])


In [12]:
def add_singleton_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Insert a size-1 dimension at position dim."""
    
    return x.unsqueeze(dim)

x3 = torch.randn(5, 7)
x3s = add_singleton_dim(x3, dim=1)

print(x3, x3.shape)
print(x3s, x3s.shape)

tensor([[-1.2203,  1.3139,  1.0533,  0.1388, -0.2044, -2.2685, -0.9133],
        [-0.4204, -0.6596, -0.7979,  0.1838,  0.2293,  0.6177, -0.2876],
        [ 0.8218,  0.1512, -0.0444,  1.6236, -2.3229, -1.7472,  1.7228],
        [ 0.7738,  0.4046, -1.6461,  1.0720,  1.5026, -0.8190, -1.3729],
        [-0.1281, -1.2838, -0.2901,  1.2767, -0.9948,  1.2176, -0.2282]]) torch.Size([5, 7])
tensor([[[-1.2203,  1.3139,  1.0533,  0.1388, -0.2044, -2.2685, -0.9133]],

        [[-0.4204, -0.6596, -0.7979,  0.1838,  0.2293,  0.6177, -0.2876]],

        [[ 0.8218,  0.1512, -0.0444,  1.6236, -2.3229, -1.7472,  1.7228]],

        [[ 0.7738,  0.4046, -1.6461,  1.0720,  1.5026, -0.8190, -1.3729]],

        [[-0.1281, -1.2838, -0.2901,  1.2767, -0.9948,  1.2176, -0.2282]]]) torch.Size([5, 1, 7])


In [13]:
def remove_singleton_dims(x: torch.Tensor, dim: int | None = None) -> torch.Tensor:
    """Remove size-1 dimensions."""
    
    return x.squeeze() if dim is None else x.squeeze(dim)

x4 = torch.randn(2, 1, 3)
x4s = remove_singleton_dims(x4)

print(x4)
print(x4s)

tensor([[[-0.0380, -2.7250, -1.8126]],

        [[-2.2068,  1.3399, -0.4337]]])
tensor([[-0.0380, -2.7250, -1.8126],
        [-2.2068,  1.3399, -0.4337]])


In [14]:
def transpose_last_two(x: torch.Tensor) -> torch.Tensor:
    """Swap the last two dimensions of x."""
    
    return x.transpose(-1, -2) #for last two -1, -2 

x6 = torch.randn(2, 3, 4)
x6t = transpose_last_two(x6)

print(x6, x6.shape)
print(x6t, x6t.shape)

tensor([[[-0.2830,  0.4928, -0.0141, -0.2747],
         [-0.7641, -0.5872,  1.1952, -1.2096],
         [-0.8989,  0.8138,  0.6532,  0.6557]],

        [[-1.4056, -1.2743,  0.4513, -0.2280],
         [-0.2201,  0.8566,  0.6465,  1.2782],
         [ 2.5501, -0.3018, -0.6703, -0.6171]]]) torch.Size([2, 3, 4])
tensor([[[-0.2830, -0.7641, -0.8989],
         [ 0.4928, -0.5872,  0.8138],
         [-0.0141,  1.1952,  0.6532],
         [-0.2747, -1.2096,  0.6557]],

        [[-1.4056, -0.2201,  2.5501],
         [-1.2743,  0.8566, -0.3018],
         [ 0.4513,  0.6465, -0.6703],
         [-0.2280,  1.2782, -0.6171]]]) torch.Size([2, 4, 3])


In [15]:
def permute_bhwc_to_bchw(x: torch.Tensor) -> torch.Tensor:
    """Convert (B, H, W, C) tensor into (B, C, H, W)."""
    
    return x.permute(0, 3, 1, 2)

x7 = torch.randn(8, 32, 32, 3)
x7p = permute_bhwc_to_bchw(x7)

print(x7)
print(x7p)

tensor([[[[ 1.6834e+00,  5.6634e-01,  1.0306e+00],
          [-3.0471e-01,  1.6873e+00,  6.8508e-01],
          [ 2.0024e+00, -5.4688e-01, -1.2076e+00],
          ...,
          [ 2.1181e-01, -7.2998e-02, -1.0638e+00],
          [-3.0500e-01,  1.2674e-01,  1.6921e+00],
          [-1.0944e+00, -1.0197e+00, -5.3986e-01]],

         [[ 7.5940e-01,  1.3026e+00, -4.2878e-01],
          [ 1.1644e-01,  1.1851e+00,  2.3862e-01],
          [ 1.4106e-01, -1.3354e+00, -4.1139e-02],
          ...,
          [-8.6139e-01, -6.6528e-01,  3.9397e-01],
          [-2.0565e+00,  1.1062e+00,  4.5620e-01],
          [ 1.4419e-02, -6.4115e-01,  2.3902e+00]],

         [[ 8.9666e-01,  1.3006e-01,  1.0874e+00],
          [-1.6537e+00, -9.8389e-01,  2.1065e+00],
          [ 5.5087e-01, -2.9364e-01, -1.5768e+00],
          ...,
          [ 2.7133e-01, -2.5230e+00,  2.5609e-01],
          [ 1.0097e+00, -4.5433e-01,  7.9665e-01],
          [-1.0394e+00, -3.2257e-01,  7.2264e-01]],

         ...,

         [[ 1.23

In [16]:
def make_contiguous(x: torch.Tensor) -> torch.Tensor:
    """Check if tensor is contiguous and if not make contiguous."""

    return x if x.is_contiguous() else x.contiguous()

x8 = torch.randn(4, 6)[:, ::2]
x8c = make_contiguous(x8)

## Indexing
Now that we know how to create tensors and manipulate them we need to understand how we can extract certain components from them using indexing. 
- Basic slicing (`x[a:b]`) returns a view when possible.
- “Fancy” indexing (lists/tensors of indices) usually allocates a new tensor.
- In-place vs out-of-place matters: if a function says “return a copy, leave the input unchanged”, you need `clone()`.

In [17]:
def slice_rows(x: torch.Tensor, start: int, end: int) -> torch.Tensor:
    """Slice rows in a 2D tensor: x[start:end, :]."""
    
    return x[start:end, :]

x = torch.arange(12).reshape(4, 3)
rows = slice_rows(x, 1, 3)

print(x)
print(rows)

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[3, 4, 5],
        [6, 7, 8]])


In [18]:
def select_columns(x: torch.Tensor, cols: Sequence[int]) -> torch.Tensor:
    """Select specific columns from a 2D tensor."""
    
    return x[:, cols]

cols = select_columns(x, [0, 2])

print(x)
print(cols)

tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[ 0,  2],
        [ 3,  5],
        [ 6,  8],
        [ 9, 11]])


In [19]:
def get_diagonal(x: torch.Tensor) -> torch.Tensor:
    """Get the diagonal of a 2D tensor."""
    
    return x.diagonal()

d = get_diagonal(torch.tensor([[1, 2], [3, 4]]))

print(d)

tensor([1, 4])


In [20]:
def set_subtensor(x: torch.Tensor, row_idx: int, col_idx: int, value: float) -> torch.Tensor:
    """Return a copy of x where x[row_idx, col_idx] is set to value."""
    
    c = x.clone()
    c[row_idx, col_idx] = value
    return c

base = torch.zeros(2, 2)
out = set_subtensor(base, 0, 1, 5.0)

print(base)
print(out)

tensor([[0., 0.],
        [0., 0.]])
tensor([[0., 5.],
        [0., 0.]])


In [21]:
def gather_rows(x: torch.Tensor, row_indices: torch.Tensor) -> torch.Tensor:
    """Gather (concat) rows from x using row_indices."""
    
    return x[row_indices, :]

x2 = torch.tensor([[10, 11], [20, 21], [30, 31]])
idx = torch.tensor([2, 0])
gathered = gather_rows(x2, idx)

print(gathered)

tensor([[30, 31],
        [10, 11]])


## Broadcasting and reducing
Now we're covering a pytorch mechanism that lets you apply elementwise ops without using python loops. It's important to understand how it works to trace your shapes in complicated systems. The broadcasting rules to know are:
- Dimensions align from the **right**.
- A dimension can broadcast if it’s equal or one of them is **1**.

### Reduction ops and `keepdim`

When you reduce over a dimension (e.g. `sum`, `mean`, `max`), PyTorch can either:

- **remove** the reduced dimension (`keepdim=False`, default), or
- **keep** it as size 1 (`keepdim=True`)

Keeping the dimension is often helpful because it makes broadcasting back “just work”.

#### Shape diagram examples

Assume `x` has shape `(B, T, D)`:

**Sum over time**
- `x.sum(dim=1)` → shape `(B, D)`
- `x.sum(dim=1, keepdim=True)` → shape `(B, 1, D)`

**Mean over features**
- `x.mean(dim=2)` → shape `(B, T)`
- `x.mean(dim=2, keepdim=True)` → shape `(B, T, 1)`

#### Why `keepdim=True` helps with broadcasting

Example: center `x` by subtracting the mean over `T`

- If `m = x.mean(dim=1)` has shape `(B, D)`, then `x - m` **fails** (shapes `(B,T,D)` and `(B,D)` don't align).
- If `m = x.mean(dim=1, keepdim=True)` has shape `(B,1,D)`, then `x - m` **works** via broadcasting.

In [22]:
def sum_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Sum tensor values along dimension dim."""
    
    return x.sum(dim=dim, keepdim=keepdim)

x = torch.ones(2, 3)
y = sum_over_dim(x, dim=1)

print(x)
print(y)

tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([3., 3.])


In [23]:
def mean_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Mean along dimension dim."""
    
    return x.mean(dim=dim, keepdim=keepdim)

x2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y2 = mean_over_dim(x2, dim=0)

print(x2)
print(y2)

tensor([[1., 2.],
        [3., 4.]])
tensor([2., 3.])


In [24]:
def max_over_dim(x: torch.Tensor, dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Max values and argmax indices along dimension dim."""
    
    return x.max(dim=dim)

x3 = torch.tensor([[1.0, 5.0], [3.0, 2.0]])
values, idx = max_over_dim(x3, dim=1)

print(values, idx)

tensor([5., 3.]) tensor([1, 0])


In [25]:
def argmax_over_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Argmax indices along dimension dim."""
    
    return x.argmax(dim=dim)

idx2 = argmax_over_dim(x3, dim=1)

print(idx2)

tensor([1, 0])


In [26]:
def broadcast_add_vector(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Add a vector v to each row of a 2D tensor x using broadcasting."""
    
    return x+v #broadcasting matches from the right!!!!
    
x4 = torch.zeros(3, 2)
v = torch.tensor([10.0, 20.0])
y4 = broadcast_add_vector(x4, v)

print(x4)
print(v)
print(y4)

tensor([[0., 0.],
        [0., 0.],
        [0., 0.]])
tensor([10., 20.])
tensor([[10., 20.],
        [10., 20.],
        [10., 20.]])


## Vectorization
We want to avoid slow (due to per-iteration overhead) python loops as much as possible and pytorch gives us many tools to avoid it. We cover these basics:
- `cat` vs `stack` (concatenate existing dims vs create a new dim)
- `repeat` vs `expand`
- `scatter_add` / `index_add` for accumulation
- `where` for conditional selection

### `expand` vs `repeat`

- `repeat(...)` **copies** data → larger tensor with independent storage.
- `expand(...)` **does not copy** data → it creates a *view* with clever strides.

This has two important implications:

1) `expand` only works when expanding a **size-1 dimension** (broadcasting a singleton).
2) The expanded tensor may have **many positions pointing to the same memory**.  
   Modifying the expanded tensor can therefore produce surprising results (multiple rows change).

Rule of thumb:
- Use `expand` for read-only broadcasting.
- Use `repeat` if you truly need independent copies.


NOTE: We implore you to write your own quick checks from now on for calling the functions and checking their output. As before you are still required to fill in the TODOs in each function.

In [27]:
def concat_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Concatenate tensors along dim. NOTE: This will always allocate new memory"""
    
    return torch.cat(tensors, dim=dim)

#check
a = torch.ones(2,3)
b = torch.zeros(2,3)
test = concat_tensors([a,b],  dim = 0)

print(test)
print(test.shape)

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [0., 0., 0.],
        [0., 0., 0.]])
torch.Size([4, 3])


In [28]:
def stack_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Stack tensors along a new dimension dim."""
    
    return torch.stack(tensors, dim=dim)

#check
a = torch.ones(2,3)
b = torch.zeros(2,3)
test = stack_tensors([a, b],  dim = 0)

print(test)
print(test.shape) #dim = 0 is the new dim we stack on

tensor([[[1., 1., 1.],
         [1., 1., 1.]],

        [[0., 0., 0.],
         [0., 0., 0.]]])
torch.Size([2, 2, 3])


In [29]:
def repeat_tensor(x: torch.Tensor, repeats: Sequence[int]) -> torch.Tensor:
    """Repeat tensor along each dimension."""
   
    return x.repeat(*repeats) #* to unpack tuplel

#check
x = torch.tensor([[6, 9]])
print(repeat_tensor(x, (3, 4, 2))) #sequence for dims to repeat


tensor([[[6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9]],

        [[6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9]],

        [[6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9],
         [6, 9, 6, 9]]])


In [30]:
def expand_tensor(x: torch.Tensor, *sizes: int) -> torch.Tensor:
    """Expand tensor to a larger size without copying data.(Sizes can be -1 to keep original dimension.)"""
    
    return x.expand(*sizes)

#check
x = torch.tensor([[1], [2], [3]]) 
y = expand_tensor(x, (3, 4))
print(y)
print(y.shape) 


tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3]])
torch.Size([3, 4])


In [31]:
def cumsum_over_dim(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Cumulative sum along dim."""
    
    return x.cumsum(dim = dim)

#check
x = torch.tensor([1, 2, 3])
print(cumsum_over_dim(x))


tensor([1, 3, 6])


In [32]:
def where_select(mask: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Elementwise select: return a where mask is True else b. mask must be broadcastable to a and b."""
    
    return torch.where(mask, a, b)

#check
mask = torch.tensor([True, False, False])
a = torch.tensor([1, 1, 0])
b = torch.tensor([0, 0, 1])
print(where_select(mask, a, b))  


tensor([1, 0, 1])


In [33]:
import torch.nn.functional as F #import for onehot

def one_hot(indices: torch.Tensor, num_classes: int, dtype: torch.dtype | None = None) -> torch.Tensor:
    """
    Create one-hot encodings.
    Output is a tensor of the same shape as indices with an added dimension of size num_classes at the end, 
    where the value along that dimension is 1 if it matches the index and 0 otherwise.

    Shapes:
    - indices: (...,) integer tensor
    Return:
    - out: (..., num_classes)

    Requirements:
    - Must work for arbitrary leading shape.
    - No Python loops.
    """

    out = F.one_hot(indices, num_classes=num_classes)
    return out if dtype is None else out.to(dtype)

#check
idx = torch.tensor([0, 2, 1, 3])
print(one_hot(idx, 4))

tensor([[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]])


In [34]:
def scatter_add_1d(
    values: torch.Tensor, indices: torch.Tensor, size: int
) -> torch.Tensor:
    """
    Sum `values` into an output vector at positions `indices`.

    Shapes:
    - values: (N,)
    - indices: (N,) integer indices in [0, size)
    Return:
    - out: (size,) with same dtype and device as values

    Requirement:
    - no Python loops
    """
    
    out = torch.zeros(size, dtype=values.dtype, device=values.device)
    out.scatter_add_(0, indices, values)
    return out

#check
values = torch.tensor([1.0, 2.0, 3.0])
indices = torch.tensor([0, 2, 2])
print(scatter_add_1d(values, indices, size=5))


tensor([1., 0., 5., 0., 0.])


In [35]:
def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """
    Count token occurrences per batch item.

    Shapes:
    - tokens: (B, T) int64
    Return:
    - counts: (B, vocab_size) where counts[b, v] = number of times token v appears in tokens[b] 

    Requirements:
    - No Python loops over B or T.
    """

    one_hot = F.one_hot(tokens, num_classes=vocab_size)
    print(one_hot) #just for me to see how it looks 
    return one_hot.sum(dim=1)

#check
tokens = torch.tensor([[0, 1, 1, 3],
                       [2, 2, 0, 1]])
counts = batched_token_histogram(tokens, vocab_size=4)
print(counts)

tensor([[[1, 0, 0, 0],
         [0, 1, 0, 0],
         [0, 1, 0, 0],
         [0, 0, 0, 1]],

        [[0, 0, 1, 0],
         [0, 0, 1, 0],
         [1, 0, 0, 0],
         [0, 1, 0, 0]]])
tensor([[1, 2, 0, 1],
        [1, 1, 2, 0]])


In [36]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Mean over `dim` considering only mask==True entries.

    Convention:
    - mask: bool tensor broadcastable to x
    - mask==True means "keep this entry"

    Return: same shape as x.mean(dim=dim)

    Requirements:
    - Avoid division by zero: if all mask are False along `dim`, define mean as 0.
    """
    
    mask_f = mask.to(dtype=x.dtype) #bool to float
    summed = (x * mask_f).sum(dim=dim)
    count = mask_f.sum(dim=dim) 
    mean = summed / count
    return torch.where(count == 0, torch.zeros_like(mean), mean) #handle 0 count

#check
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
mask = torch.tensor([[True, False], [False, False]])
print(masked_mean(x, mask, dim=1))


tensor([1., 0.])


## Einsum warmup
Now that you’re comfortable with shapes and broadcasting, we’ll introduce `torch.einsum`, a concise way to express tensor operations by explicitly naming axes and summing over repeated indices.

### The idea
You describe each input tensor by labeling its dimensions with letters, e.g.
- `x: (B, T, D)` → `"btd"`
- `W: (D, H)`    → `"dh"`

Then you tell einsum what output labels you want:
- `"btd,dh->bth"`

### Rules of einsum
1) **Same letter = same axis** (must match in size, except broadcastable size-1).
2) **Repeated letters are summed over** (a “contraction”).
3) **Letters that appear in the output are kept** (in that order).
4) You can **reorder axes** just by changing the output label order.

### Tiny cheat sheet
- Sum over an axis: `"btd->bt"` (sums over `d`)
- Transpose: `"ij->ji"`
- Dot product: `"d,d->"` or batched `"btd,btd->bt"`
- Matrix multiply: `"ik,kj->ij"`
- Batched matmul: `"bij,bjk->bik"`
- Outer product: `"i,j->ij"`

### How to derive an einsum (recommended workflow)
1) Write down shapes with named axes (e.g. `q: b h t d`, `k: b h s d`).
2) Decide which axes you want to **sum over** (give them the same letter in both inputs).
3) Decide which axes you want to **keep** in the output (write them after `->`).

In this section, you’ll use einsum to implement building blocks that show up in attention:
- linear projections (`x @ W`)
- dot products
- attention score matrices (`QKᵀ`)
- applying attention weights (`softmax(scores) @ V`)

NOTE: For these exercises you are required to use `torch.einsum` not `matmul` (we check). You are also not required to understand the attention mechanism at this point and the exercises are sovable without. It is good however, to remember the implementations in this exercise for future implementations.

In [37]:
def einsum_linear_btd_dh_to_bth(x: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """
    Linear projection using einsum.

    Shapes:
    - x: (B, T, D)
    - W: (D, H)
    Return:
    - y: (B, T, H)
    """
    return torch.einsum('btd,dh->bth', x, W)

# --- checks
import torch
B, T, D, H = 2, 3, 4, 5
x = torch.randn(B, T, D)
W = torch.randn(D, H)
y = einsum_linear_btd_dh_to_bth(x, W)
print('einsum_linear_btd_dh_to_bth', y.shape)


einsum_linear_btd_dh_to_bth torch.Size([2, 3, 5])


In [38]:
def einsum_pairwise_dot(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Pairwise dot product between x and y.

    Shapes:
    - x: (B, T, D)
    - y: (B, T, D)
    Return:
    - dots: (B, T) where dots[b,t] = dot(x[b,t], y[b,t])
    """
    return torch.einsum('btd,btd->bt', x, y)

# --- checks
import torch
B, T, D = 2, 3, 4
x = torch.randn(B, T, D)
y = torch.randn(B, T, D)
dots = einsum_pairwise_dot(x, y)
print('einsum_pairwise_dot', dots.shape, 'sample', dots[0, 0].item())


einsum_pairwise_dot torch.Size([2, 3]) sample 0.48807650804519653


In [39]:
def einsum_qk_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Compute attention scores QK^T using einsum.

    Shapes:
    - q: (B, H, T, Dh)
    - k: (B, H, T, Dh)
    Return:
    - scores: (B, H, T, T) where scores[b,h,i,j] = dot(q[b,h,i], k[b,h,j])
    """
    return torch.einsum('bhtd,bhsd->bhts', q, k)

# --- checks
import torch
B, H, T, Dh = 2, 3, 4, 5
q = torch.randn(B, H, T, Dh)
k = torch.randn(B, H, T, Dh)
scores = einsum_qk_scores(q, k)
print('einsum_qk_scores', scores.shape)


einsum_qk_scores torch.Size([2, 3, 4, 4])


In [40]:
def einsum_apply_attention(weights: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Apply attention weights to values using einsum.

    Shapes:
    - weights: (B, H, T, T)
    - v:       (B, H, T, Dh)
    Return:
    - out:     (B, H, T, Dh) where out[b,h,i] = sum_j weights[b,h,i,j] * v[b,h,j]
    """
    return torch.einsum('bhts,bhsd->bhtd', weights, v)

# --- checks
import torch
B, H, T, Dh = 2, 3, 4, 5
weights = torch.softmax(torch.randn(B, H, T, T), dim=-1)
v = torch.randn(B, H, T, Dh)
out = einsum_apply_attention(weights, v)
print('einsum_apply_attention', out.shape)


einsum_apply_attention torch.Size([2, 3, 4, 5])


## Attention Fundamentals
This exercise introduces some building blocks of the attention mechanism which we will encounter extensively throughout the course. It's not yet required for you to fully understand the mechanism to implement the exercises. However, it's good to remember these building blocks for the future. 

To complete the exercises you should familiarize yourself with these topics:
- Stable softmax read: https://jaykmody.com/blog/stable-softmax/
- Masking: typically this means setting masked logits to -inf *before* softmax.
- For attention: causal masks are upper-triangular (no attending to the future).

In [41]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Numerically stable softmax along `dim`.

    Requirements:
    - Must not overflow for large values in x.
    - Output sums to 1 along `dim`.
    """
    max_x = torch.max(x, dim=dim, keepdim=True).values
    x_shifted = x - max_x
    exp_x = torch.exp(x_shifted)
    sum_exp = exp_x.sum(dim=dim, keepdim=True)
    return exp_x / sum_exp

# --- checks
import torch
logits = torch.randn(2, 4) * 10
sm = stable_softmax(logits, dim=-1)
print('stable_softmax', sm, 'row sums', sm.sum(dim=-1))


stable_softmax tensor([[6.6681e-11, 1.7799e-15, 1.3021e-04, 9.9987e-01],
        [9.9981e-01, 8.3796e-11, 1.8225e-04, 9.4690e-06]]) row sums tensor([1.0000, 1.0000])


In [42]:
def masked_fill_tensor(x: torch.Tensor, mask: torch.Tensor, value: float) -> torch.Tensor:
    """
    Return a copy of x where positions with mask == True are replaced by `value`.
    
    Requirements:
    - mask must be broadcastable to x.
    - do NOT modify x in-place.
    """
    return x.masked_fill(mask, value)

# --- checks
import torch
x = torch.randn(2, 4)
mask = torch.tensor([[False, True, False, True], [True, False, False, True]])
y = masked_fill_tensor(x, mask, -1e9)
print('masked_fill_tensor', y)


masked_fill_tensor tensor([[-1.2990e+00, -1.0000e+09, -7.7028e-03, -1.0000e+09],
        [-1.0000e+09, -2.3488e-01,  1.6405e-02, -1.0000e+09]])


In [43]:
def masked_softmax(x: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Softmax over x with a boolean mask.

    Convention:
    - mask == True means "invalid and must receive probability 0".
    - Do masking before softmax (i.e., set invalid logits to a large negative).”

    Requirements:
    - Must be numerically stable.
    - Output must be exactly 0 where mask==True.
    - If all entries are masked along `dim`, return all zeros along `dim`.
    - You may reuse functions you implemented above.
    """
    x_masked = masked_fill_tensor(x, mask, -1e9)
    probs = stable_softmax(x_masked, dim=dim)
    probs = torch.where(mask, torch.zeros_like(probs), probs)

    all_masked = mask.all(dim=dim, keepdim=True)
    probs = torch.where(all_masked, torch.zeros_like(probs), probs)
    return probs

# --- checks
import torch
logits = torch.randn(2, 4) * 10
mask = torch.tensor([[False, True, False, True], [True, True, True, True]])
msm = masked_softmax(logits, mask, dim=-1)
print('masked_softmax', msm, 'row sums', msm.sum(dim=-1))


masked_softmax tensor([[0.9978, 0.0000, 0.0022, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]]) row sums tensor([1., 0.])


In [45]:
def make_causal_mask(T: int, device: torch.device | str | None = None) -> torch.Tensor:
    """
    Create a causal (future-masking) boolean mask of shape (T, T).

    Convention:
    - mask[i, j] == True  => position (i attends to j) is NOT allowed (j is in the future)
    - mask[i, j] == False => allowed

    So this is an upper-triangular mask above the diagonal.

    Return:
    - mask: boolean tensor on the specified device

    Example (T=4):
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]
    """
    return torch.triu(torch.ones(T, T, dtype=torch.bool, device=device), diagonal=1)

# --- checks
cm = make_causal_mask(4)
print('make_causal_mask', cm)


make_causal_mask tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])


In [46]:
def apply_causal_mask(attn_logits: torch.Tensor, value: float = -1e9) -> torch.Tensor:
    """
    Apply a causal mask to attention logits.

    Expected shapes:
    - attn_logits: (..., T, T)

    Returns:
    - masked logits (same shape) where masked positions have been set to `value`.

    Notes:
    - Create a causal mask for the final two dims.
    - Broadcast it across leading dims.
    - You may reuse functions declared above.
    """
    T = attn_logits.size(-1)
    mask = make_causal_mask(T, device=attn_logits.device)
    return masked_fill_tensor(attn_logits, mask, value)

# --- checks
import torch
attn_logits = torch.arange(16.0).view(1, 4, 4)
masked_logits = apply_causal_mask(attn_logits)
print('apply_causal_mask', masked_logits)


apply_causal_mask tensor([[[ 0.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09],
         [ 4.0000e+00,  5.0000e+00, -1.0000e+09, -1.0000e+09],
         [ 8.0000e+00,  9.0000e+00,  1.0000e+01, -1.0000e+09],
         [ 1.2000e+01,  1.3000e+01,  1.4000e+01,  1.5000e+01]]])
